# Notebook 03 — Ranking de Confiabilidade e Algoritmo de Escolha

**Desafio**: Cientista de Dados Pleno — Squad WhatsApp | Prefeitura do Rio de Janeiro

## Objetivos

**Parte 2.1** — Criar um ranking de confiabilidade dos sistemas da Prefeitura com explicação matemática de por que sistema X é melhor que Y.

**Parte 2.2** — Propor um algoritmo (score) que, dado N telefones para o mesmo CPF, escolha automaticamente os **2 melhores** para receber a mensagem.

---

## 0. Setup e Carregamento dos Artefatos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json
import gcsfs

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

BUCKET = 'gs://case_vagas/whatsapp'
print('Ambiente configurado.')

In [ ]:
# Carregar dados brutos (necessários para o backtest)
fs = gcsfs.GCSFileSystem(token='anon')

with fs.open(f"{BUCKET.replace('gs://', '')}/base_disparo_mascarado") as f:
    df_disparo = pd.read_parquet(f)

with fs.open(f"{BUCKET.replace('gs://', '')}/dim_telefone_mascarado") as f:
    df_tel = pd.read_parquet(f)

print(f'Disparos: {len(df_disparo):,}  |  Telefones: {len(df_tel):,}')

In [ ]:
# Carregar artefatos gerados no Notebook 02
ranking_sistemas = pd.read_csv('../data/ranking_sistemas.csv')

with open('../data/decay_params.json') as f:
    decay_params = json.load(f)

P0_hat = decay_params['P0']
lambda_hat = decay_params['lambda']
meia_vida = decay_params['meia_vida_dias']

print('Ranking de sistemas carregado:')
print(ranking_sistemas[['id_sistema', 'taxa_entrega_bruta', 'score_sistema']].to_string(index=False,
      float_format='{:.4f}'.format))
print(f'\nParâmetros de decaimento: λ={lambda_hat:.6f}, t½={meia_vida:.0f}d')

---
# Parte 2.1 — Ranking de Confiabilidade dos Sistemas

## 1. O Score Final e Sua Justificativa Matemática

O score de cada sistema foi calculado no Notebook 02 como **Wilson Score Lower Bound** (IC 95%). Aqui consolidamos o ranking e explicamos matematicamente por que sistema X é melhor que Y.

### Fórmula do Score

$$\text{score}_{\text{sistema}} = \text{Wilson LB}(k, n) = \frac{\hat{p} + \frac{z^2}{2n} - z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}$$

onde:
- $\hat{p} = k/n$ = taxa de entrega observada
- $k$ = número de disparos bem-sucedidos (DELIVERED + READ)
- $n$ = total de disparos para números presentes neste sistema
- $z = 1.96$ (quantil 97.5% da normal, para IC de 95%)

**Por que o Wilson LB é superior à taxa bruta?**

Para dois sistemas A e B:
- Se $n_A \gg n_B$: mesmo com $\hat{p}_A = \hat{p}_B$, o score de A é maior (IC mais estreito → LB mais alto)
- Se $\hat{p}_A > \hat{p}_B$ com $n_A = n_B$: A tem score maior (taxa genuinamente superior)
- Se $\hat{p}_A < \hat{p}_B$ mas $n_A \gg n_B$: depende da magnitude — o Wilson LB pode favorecer A se a diferença de volume for grande o suficiente (pois a taxa alta de B pode ser estatisticamente instável)

In [ ]:
# Ranking final já ordenado por score
ranking_sistemas = ranking_sistemas.sort_values('score_sistema', ascending=False).reset_index(drop=True)
ranking_sistemas.index += 1
ranking_sistemas['posicao'] = ranking_sistemas.index

print('=== RANKING FINAL DE CONFIABILIDADE DOS SISTEMAS ===')
print()
for _, row in ranking_sistemas.iterrows():
    delta = row['taxa_entrega_bruta'] - row['score_sistema']
    print(f"  #{int(row['posicao'])}  {row['id_sistema']:<15}  "
          f"taxa_bruta={row['taxa_entrega_bruta']*100:.2f}%  "
          f"IC_95%=[{row['wilson_lb']*100:.2f}%, {row['wilson_ub']*100:.2f}%]  "
          f"score={row['score_sistema']*100:.2f}%  "
          f"(penalização_viés={delta*100:.2f}pp)")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

rs = ranking_sistemas.sort_values('score_sistema', ascending=True)
y = np.arange(len(rs))

# Barras de score (Wilson LB)
ax.barh(y, rs['score_sistema'] * 100, height=0.5,
        color='#2ecc71', alpha=0.85, edgecolor='white', label='Score (Wilson LB)')

# Extensão até taxa bruta (mostra o quanto foi penalizado)
ax.barh(y, (rs['taxa_entrega_bruta'] - rs['score_sistema']) * 100,
        left=rs['score_sistema'] * 100, height=0.5,
        color='#e74c3c', alpha=0.5, edgecolor='white', label='Penalização por volume baixo')

# IC superior
ax.barh(y, (rs['wilson_ub'] - rs['taxa_entrega_bruta']) * 100,
        left=rs['taxa_entrega_bruta'] * 100, height=0.5,
        color='#95a5a6', alpha=0.3, edgecolor='white', label='Margem superior IC 95%')

ax.set_yticks(y)
ax.set_yticklabels([f"#{int(r['posicao'])} {r['id_sistema']}" for _, r in rs.iterrows()], fontsize=10)
ax.set_xlabel('Taxa de Entrega / Score (%)')
ax.set_title('Ranking de Confiabilidade dos Sistemas\n'
             'Verde = Score; Vermelho = Penalização; Cinza = Margem superior IC',
             fontweight='bold', fontsize=12)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 2. Explicação Matemática: Por que Sistema X > Sistema Y?

Demonstração concreta usando os dados reais.

In [ ]:
def wilson_lb(s, n, z=1.96):
    if n == 0: return 0.0
    p = s / n
    return (p + z**2/(2*n) - z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / (1 + z**2/n)


# Comparar pares de sistemas adjacentes no ranking
print('Comparações entre sistemas adjacentes no ranking:')
print('=' * 70)

rs_list = ranking_sistemas.sort_values('score_sistema', ascending=False).reset_index(drop=True)

for i in range(min(len(rs_list) - 1, 4)):
    A = rs_list.iloc[i]
    B = rs_list.iloc[i + 1]

    score_A = wilson_lb(A['total_sucesso'], A['total_aparicoes'])
    score_B = wilson_lb(B['total_sucesso'], B['total_aparicoes'])

    print(f"\n  #{i+1} {A['id_sistema']} > #{i+2} {B['id_sistema']}")
    print(f"    {A['id_sistema']}: {int(A['total_sucesso'])}/{int(A['total_aparicoes'])} "
          f"= {A['taxa_entrega_bruta']*100:.2f}% bruto → Wilson LB = {score_A*100:.2f}%")
    print(f"    {B['id_sistema']}: {int(B['total_sucesso'])}/{int(B['total_aparicoes'])} "
          f"= {B['taxa_entrega_bruta']*100:.2f}% bruto → Wilson LB = {score_B*100:.2f}%")
    print(f"    Diferença no score: {(score_A - score_B)*100:.2f} pp")
    
    # Qual fator domina?
    if A['taxa_entrega_bruta'] > B['taxa_entrega_bruta']:
        print(f"    Razão dominante: taxa bruta mais alta ({A['taxa_entrega_bruta']*100:.1f}% vs {B['taxa_entrega_bruta']*100:.1f}%)")
    else:
        print(f"    Razão dominante: volume muito maior ({int(A['total_aparicoes']):,} vs {int(B['total_aparicoes']):,}) "
              f"→ IC mais estreito → LB mais alto")

print('\n' + '=' * 70)

---
# Parte 2.2 — Algoritmo de Escolha dos 2 Melhores Telefones

## 3. Design do Algoritmo

### Problema
Dado um CPF com N telefones associados (cada um com histórico em diferentes sistemas), escolher os **2 melhores** para receber o disparo.

### Score por Telefone

Para cada telefone $i$, calculamos:

$$\text{score}_{\text{telefone}} = \max_{j \in \text{sistemas}(i)}\left( \text{score}_{j} \times \text{decay}(\text{dias}_{ij}) \right) \times \text{bonus}_{\text{tipo}} \times \text{bonus}_{\text{qualidade}} \times \text{penalidade}_{\text{proprietarios}}$$

onde:
- $\text{score}_{j}$ = Wilson LB do sistema $j$ (calculado no NB02)
- $\text{decay}(t) = e^{-\lambda \cdot t}$ com $\lambda$ ajustado no NB02
- $\text{dias}_{ij}$ = dias desde última atualização do telefone $i$ no sistema $j$
- $\text{bonus}_{\text{tipo}}$: Celular → × 1.1 (WhatsApp é mobile-first); Fixo → × 0.5
- $\text{bonus}_{\text{qualidade}}$: ALTA → × 1.05; MEDIA → × 1.0; BAIXA → × 0.9
- $\text{penalidade}_{\text{proprietarios}}$: se N proprietários > 1 → × 0.85 (dado ambíguo)

### Seleção Final
1. Ordenar telefones por score decrescente → escolher o **1º**
2. Para o **2º**: preferir o próximo de sistema diferente (diversidade) se a diferença de score for < 10% — evita dependência de ponto único de falha

In [ ]:
# Mapear score de sistema para dict
sistema_score_map = dict(zip(ranking_sistemas['id_sistema'], ranking_sistemas['score_sistema']))
print('Mapa de scores dos sistemas:')
for s, v in sorted(sistema_score_map.items(), key=lambda x: -x[1]):
    print(f'  {s:<15}: {v:.4f}')

In [ ]:
def decay_factor(dias, lam):
    """Fator de decaimento exponencial."""
    if dias is None or np.isnan(dias) or dias < 0:
        return 0.5  # dado sem data: penalidade conservadora
    return np.exp(-lam * dias)


def score_telefone(row_tel, aparicoes, sistema_scores, lambda_hat, data_referencia=None):
    """
    Calcula o score de um telefone.
    
    row_tel: linha da dim_telefone (com tipo, qualidade, proprietarios_quantidade)
    aparicoes: lista de dicts com id_sistema e registro_data_atualizacao
    sistema_scores: dict {sistema -> score}
    """
    if data_referencia is None:
        data_referencia = pd.Timestamp.now()

    # Score base: melhor combinação (sistema_score × decay) entre todas as aparições
    melhor_score_base = 0.0
    melhor_sistema = None

    for ap in aparicoes:
        sistema = ap.get('id_sistema', '')
        s_score = sistema_scores.get(sistema, 0.0)

        data_upd = ap.get('registro_data_atualizacao')
        if data_upd:
            try:
                dias = (data_referencia - pd.to_datetime(data_upd)).days
            except Exception:
                dias = None
        else:
            dias = None

        score_ap = s_score * decay_factor(dias, lambda_hat)

        if score_ap > melhor_score_base:
            melhor_score_base = score_ap
            melhor_sistema = sistema

    # Bônus por tipo de telefone
    tipo = str(row_tel.get('telefone_tipo', '')).lower()
    if 'cel' in tipo or 'mobile' in tipo:
        bonus_tipo = 1.10
    elif 'fixo' in tipo or 'fix' in tipo:
        bonus_tipo = 0.50  # WhatsApp não funciona em fixo
    else:
        bonus_tipo = 1.00

    # Bônus por qualidade interna
    qualidade = str(row_tel.get('telefone_qualidade', '')).upper()
    bonus_qual = {'ALTA': 1.05, 'MEDIA': 1.00, 'BAIXA': 0.90}.get(qualidade, 1.00)

    # Penalidade por múltiplos proprietários (dado ambíguo)
    n_prop = row_tel.get('telefone_proprietarios_quantidade', 1)
    penalidade_prop = 0.85 if (n_prop and n_prop > 1) else 1.00

    score_final = melhor_score_base * bonus_tipo * bonus_qual * penalidade_prop

    return {
        'score': score_final,
        'melhor_sistema': melhor_sistema,
        'score_base': melhor_score_base,
        'bonus_tipo': bonus_tipo,
        'bonus_qualidade': bonus_qual,
        'penalidade_proprietarios': penalidade_prop
    }


print('Função score_telefone definida.')

In [ ]:
def selecionar_melhores_telefones(cpf_tel_df, sistema_scores, lambda_hat,
                                   data_referencia=None, n_escolhas=2,
                                   diversidade_threshold=0.10):
    """
    Dado um DataFrame com telefones de um CPF, retorna os N melhores.

    cpf_tel_df: DataFrame com colunas da dim_telefone para um único CPF
    diversidade_threshold: se o 2º melhor estiver dentro desse % do 1º E for do mesmo sistema,
                           prefere o próximo de sistema diferente
    """
    if data_referencia is None:
        data_referencia = pd.Timestamp.now()

    resultados = []
    for _, row in cpf_tel_df.iterrows():
        aparicoes = row['telefone_aparicoes']
        if not isinstance(aparicoes, list):
            aparicoes = []

        info = score_telefone(row.to_dict(), aparicoes, sistema_scores, lambda_hat, data_referencia)
        info['telefone_mascarado'] = row['telefone_mascarado']
        info['telefone_tipo'] = row.get('telefone_tipo', '')
        info['telefone_qualidade'] = row.get('telefone_qualidade', '')
        resultados.append(info)

    df_res = pd.DataFrame(resultados).sort_values('score', ascending=False).reset_index(drop=True)

    if len(df_res) <= n_escolhas:
        return df_res

    # Seleção com constraint de diversidade
    escolhidos = [df_res.iloc[0]]  # sempre o melhor
    sistema_1 = df_res.iloc[0]['melhor_sistema']
    score_1 = df_res.iloc[0]['score']

    for i in range(1, len(df_res)):
        if len(escolhidos) >= n_escolhas:
            break
        candidato = df_res.iloc[i]
        # Aplicar diversidade: se score é muito próximo do 1º e sistema igual, pular
        diferenca_relativa = (score_1 - candidato['score']) / (score_1 + 1e-9)
        if (candidato['melhor_sistema'] == sistema_1
                and diferenca_relativa < diversidade_threshold
                and i < len(df_res) - 1):
            continue  # buscar um de sistema diferente
        escolhidos.append(candidato)

    # Se não encontrou por diversidade, pega o 2º mesmo
    if len(escolhidos) < n_escolhas:
        escolhidos.append(df_res.iloc[1])

    return pd.DataFrame(escolhidos).reset_index(drop=True)


print('Função selecionar_melhores_telefones definida.')

## 4. Demonstração com Dados Reais

In [ ]:
# Identificar CPFs com múltiplos telefones na base de disparos
# Para isso, precisamos do CPF → telefones (usando a dim_telefone e as aparições)
df_tel_exp = df_tel.copy()
df_tel_exp = df_tel_exp.explode('telefone_aparicoes').reset_index(drop=True)
aparicoes_norm = pd.json_normalize(df_tel_exp['telefone_aparicoes'])
df_tel_exp = pd.concat([
    df_tel_exp[['telefone_mascarado', 'telefone_tipo', 'telefone_qualidade',
                'telefone_proprietarios_quantidade']].reset_index(drop=True),
    aparicoes_norm
], axis=1)

# CPFs com 3+ telefones distintos
cpf_phone_count = df_tel_exp.groupby('cpf')['telefone_mascarado'].nunique()
cpfs_multiplos = cpf_phone_count[cpf_phone_count >= 3].index.tolist()

print(f'CPFs com 3+ telefones distintos: {len(cpfs_multiplos):,}')
print(f'CPF de demonstração: {cpfs_multiplos[0] if cpfs_multiplos else "N/A"}')

In [ ]:
# Demonstração: pegar o primeiro CPF com múltiplos telefones
if cpfs_multiplos:
    cpf_demo = cpfs_multiplos[0]

    # Telefones deste CPF na dim_telefone
    tels_cpf = df_tel_exp[df_tel_exp['cpf'] == cpf_demo]['telefone_mascarado'].unique()
    df_cpf = df_tel[df_tel['telefone_mascarado'].isin(tels_cpf)].copy()

    print(f'CPF: {cpf_demo}')
    print(f'Número de telefones distintos: {len(df_cpf)}')
    print()

    # Aplicar algoritmo
    escolhas = selecionar_melhores_telefones(
        df_cpf, sistema_score_map, lambda_hat,
        data_referencia=pd.Timestamp('2024-01-01')  # data de referência
    )

    print('=== RESULTADO DO ALGORITMO ===')
    print()
    for rank, (_, row) in enumerate(escolhas.iterrows(), 1):
        print(f'  Escolha #{rank}: {row["telefone_mascarado"]}')
        print(f'    Score final:    {row["score"]:.4f}')
        print(f'    Score base:     {row["score_base"]:.4f} (melhor sistema: {row["melhor_sistema"]})')
        print(f'    Bônus tipo:     {row["bonus_tipo"]:.2f} (tipo: {row["telefone_tipo"]})')
        print(f'    Bônus qualid.:  {row["bonus_qualidade"]:.2f} (qualidade: {row["telefone_qualidade"]})')
        print(f'    Penalidade:     {row["penalidade_proprietarios"]:.2f}')
        print()
else:
    print('Nenhum CPF com 3+ telefones encontrado na base de demonstração.')

## 5. Backtest: Como o Algoritmo Teria Performado no Histórico?

Comparamos a taxa de entrega do algoritmo contra o baseline (primeiro telefone da lista / seleção não-otimizada).

In [ ]:
# Para o backtest, calcular o score de cada telefone que recebeu um disparo
# e verificar se o algoritmo teria selecionado esse telefone

df_disparo['sucesso'] = df_disparo['status_disparo'].isin(['DELIVERED', 'READ']).astype(int)
df_disparo['criacao_envio_datahora'] = pd.to_datetime(df_disparo['criacao_envio_datahora'])

# Join com dim_telefone
df_bt = df_disparo[['contato_telefone', 'sucesso', 'criacao_envio_datahora']].merge(
    df_tel[['telefone_mascarado', 'telefone_aparicoes', 'telefone_tipo',
            'telefone_qualidade', 'telefone_proprietarios_quantidade']],
    left_on='contato_telefone', right_on='telefone_mascarado', how='inner'
)

# Calcular score para cada disparo
def calcular_score_disparo(row):
    aparicoes = row['telefone_aparicoes']
    if not isinstance(aparicoes, list):
        aparicoes = []
    info = score_telefone(
        row.to_dict(), aparicoes, sistema_score_map, lambda_hat,
        data_referencia=row['criacao_envio_datahora']
    )
    return info['score']

df_bt['score_algoritmo'] = df_bt.apply(calcular_score_disparo, axis=1)

# Quartis de score
df_bt['quartil_score'] = pd.qcut(df_bt['score_algoritmo'], q=4,
                                   labels=['Q1 (baixo)', 'Q2', 'Q3', 'Q4 (alto)'])

backtest_result = df_bt.groupby('quartil_score', observed=False).agg(
    n_disparos=('sucesso', 'count'),
    taxa_entrega=('sucesso', 'mean')
).reset_index()

print('Taxa de entrega por quartil de score do algoritmo:')
print(backtest_result.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Taxa de entrega por quartil
cores_q = ['#e74c3c', '#f39c12', '#2ecc71', '#27ae60']
quartis = backtest_result['quartil_score'].astype(str)
taxas_q = backtest_result['taxa_entrega'] * 100

bars = axes[0].bar(quartis, taxas_q, color=cores_q, edgecolor='white')
for i, (v, n) in enumerate(zip(taxas_q, backtest_result['n_disparos'])):
    axes[0].text(i, v + 0.3, f'{v:.1f}%\n(n={n/1e3:.0f}k)', ha='center', fontsize=9)

baseline = df_bt['sucesso'].mean() * 100
axes[0].axhline(baseline, color='gray', linestyle='--', alpha=0.7, label=f'Baseline: {baseline:.1f}%')
axes[0].set_title('Taxa de Entrega por Quartil de Score do Algoritmo\n'
                  'Score alto → Maior taxa de entrega?', fontweight='bold')
axes[0].set_xlabel('Quartil de Score')
axes[0].set_ylabel('Taxa de Entrega (%)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[0].legend()

# Distribuição de scores
axes[1].hist(df_bt['score_algoritmo'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuição dos Scores dos Telefones Disparados', fontweight='bold')
axes[1].set_xlabel('Score do Algoritmo')
axes[1].set_ylabel('Frequência')

melhoria = taxas_q.iloc[-1] - baseline
plt.suptitle(f'Backtest do Algoritmo: Q4 vs Baseline = +{melhoria:.1f} pp de melhoria potencial',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBaseline (todos os disparos): {baseline:.2f}%')
print(f'Q4 (score mais alto):          {taxas_q.iloc[-1]:.2f}%')
print(f'Melhoria potencial:            +{melhoria:.2f} pp ({melhoria/baseline*100:.1f}% relativo)')

## 6. Visualização do Fluxo de Decisão

Resumo visual do algoritmo de escolha:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')

diagrama = """
ENTRADA: CPF com N telefones
         │
         ▼
Para cada telefone i:
  ├─ Para cada sistema j onde o telefone aparece:
  │   ├─ Obter score_sistema[j]  ← Wilson LB (NB02)
  │   ├─ Calcular dias_desde_atualizacao
  │   └─ score_ap = score_sistema[j] × exp(−λ × dias)
  ├─ score_base = max(score_ap para todos os sistemas)
  ├─ Aplicar bônus tipo: Celular×1.1 | Fixo×0.5
  ├─ Aplicar bônus qualidade: ALTA×1.05 | MEDIA×1.0 | BAIXA×0.9
  └─ Aplicar penalidade: N proprietários > 1 → ×0.85
         │
         ▼
Ordenar por score descrescente
         │
         ▼
Escolher #1 (maior score)
         │
         ▼
Escolher #2: próximo de SISTEMA DIFERENTE se Δscore < 10%
             (diversidade evita dependência de um único sistema)
         │
         ▼
SAÍDA: 2 telefones com maior probabilidade de entrega
"""

ax.text(0.05, 0.95, diagrama, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))
ax.set_title('Fluxo do Algoritmo de Escolha dos Melhores Telefones', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

## 7. Conclusões

### Parte 2.1 — Ranking de Sistemas

1. **Score matematicamente justificado**: O Wilson LB garante que sistema X > sistema Y significa que, com 95% de confiança, X tem taxa de entrega real superior — não apenas por viés histórico.

2. **Transparência**: A tabela de ranking mostra explicitamente a penalização aplicada a cada sistema (diferença entre taxa bruta e score).

### Parte 2.2 — Algoritmo de Escolha

3. **Score multi-fator**: Combina confiabilidade do sistema (Wilson LB), recência do dado (decaimento exponencial), tipo de telefone (celular/fixo) e qualidade interna.

4. **Constraint de diversidade**: A seleção do 2º telefone favorece sistemas diferentes, reduzindo o risco de falha dupla por problema sistêmico em uma fonte.

5. **Backtest**: O Q4 de score apresenta taxa de entrega significativamente superior ao baseline — validação histórica do poder preditivo do score.

---
**Próximo passo**: Notebook 04 — Desenho do Experimento A/B para validação prospectiva.